##### ***强化学习(RL, Reinforce Learning)基础***
###### 在前面的学习中我们主要接触的是监督学习和自监督学习，其中监督学习通常会为每一个输入准备一个明确的正确答案，模型通过比较预测结果与标签之间的差异来更新参数；而如GPT的Pretraining虽然不需要人工标注，但下一个token也能通过原始文本直接提供，因此本质上仍然能够构造明确的监督信号。
###### 但在现实任务中，很多问题并不存在一个唯一的正确答案。例如，当一个语言模型回答用户问题时，我们很难简单地说某一个回答是绝对正确的，往往只能判断某个回答相对于另一个回答是否更加有帮助、更加安全或者更加符合人类偏好。这时，如果仍然只依赖传统的监督学习，就很难完整描述我们真正想要的目标，于是便需要引入强化学习。

###### 强化学习的核心思想并不是直接告诉模型每一步的正确答案，而是让模型不断采取行动，并根据行动之后得到的奖励来调整策略。模型并不知道每一步应该做什么，而是通过与环境进行交互，逐渐学会哪些行为更容易获得较高的长期回报。

##### ***强化学习的基本组成***

###### 一个完整的强化学习问题通常包含智能体Agent、环境Environment、状态State、动作Action和奖励Reward。智能体是做出决策的主体，环境则是智能体所处的外部世界；状态表示智能体当前所处的情况，动作表示智能体在当前状态下采取的行为，而奖励则是环境对该行为给出的反馈。

###### 强化学习的过程可以概括为：智能体观察当前状态，根据策略选择一个动作；环境接收到动作后发生变化，并返回新的状态和奖励；智能体再根据新的反馈调整自己的策略。这个过程会不断重复，最终目标并不是让某一次动作获得最高奖励，而是让长期累计奖励尽可能大。

###### 其中，策略Policy可以理解为智能体在不同状态下选择不同动作的规则。对于一个确定性策略，给定状态后总是选择同一个动作；对于一个随机策略，给定状态后则会输出一组动作概率。现代语言模型更接近后者，因为模型并不是固定输出某一个Token，而是会为词表中的每个Token计算一个概率。

##### ***从一次决策到连续决策***

###### 如果一个任务只需要做出一次选择，那么它更像是一个简单的决策问题；但在很多实际任务中，当前动作会影响后续状态，后续状态又会影响下一次动作，这就形成了连续决策过程。强化学习不仅需要考虑当前动作带来的即时奖励，还需要考虑当前动作对未来奖励的影响。

###### 例如，在一个迷宫任务中，智能体走向某个方向可能暂时没有奖励，但这个动作可能使它更接近终点；相反，某个动作也许能够获得短期奖励，却可能让智能体陷入死路。因此，强化学习真正关心的是一系列动作带来的累计回报，而不是某一步的单独结果。

###### 在语言模型中也存在类似情况。模型每生成一个Token，都会改变后续的上下文，并影响之后能够生成的内容。因此，一段完整回答可以被看作由多个连续动作组成的轨迹，而每一个Token都是一次动作。

##### ***强化学习与语言模型的对应关系***

###### 对于语言模型来说，当前已经输入的Prompt以及模型已经生成的Token共同构成当前状态；模型根据当前状态从词表中选择下一个Token，这个Token便可以看作一次动作；当模型生成完整回答后，奖励模型或者人工评价系统会根据回答质量给出奖励。

###### 因此，强化学习中的概念可以与语言模型进行如下对应：状态对应当前上下文，动作对应下一个Token，策略对应语言模型本身，轨迹对应完整生成过程，而奖励则对应回答的质量、安全性和人类偏好。

###### 需要注意的是，语言模型并不是在每次生成Token之后都能立即得到一个明确奖励。很多时候，真正的奖励要等整段回答生成完成之后才能得到，这使得语言模型的对齐问题具有延迟奖励和长期信用分配的特点。后续的RLHF正是试图利用强化学习解决这一类问题。

##### ***最简单的强化学习问题：多臂老虎机***

###### 为了更直观地理解强化学习，我们先考虑一个只有一次决策的简单问题。假设现在有多个按钮，每个按钮被按下后都有可能得到不同的奖励，但智能体一开始并不知道哪个按钮的收益最高。它只能不断尝试不同的按钮，并根据得到的奖励逐渐判断哪个选择更好。

###### 这个问题被称为多臂老虎机问题。每个按钮都可以看作一个动作，按下按钮后得到的数值可以看作奖励，而智能体选择不同按钮的概率便构成了当前策略。与监督学习不同，这里没有人为告诉模型正确的按钮，模型只能通过不断尝试和获得反馈来调整自己的选择概率。

###### 多臂老虎机虽然没有复杂的状态转移，但它已经包含了强化学习最核心的思想：模型需要在探索未知动作和利用当前最优动作之间进行平衡。探索可以帮助模型发现更好的选择，利用则是尽可能选择当前已知收益较高的动作。

##### ***回报与价值函数***
###### 在多臂老虎机中，智能体只需要关注当前动作带来的奖励；但在连续决策问题中，当前动作的影响可能要经过很多步之后才会体现出来。因此，强化学习不能只关注某一步的即时奖励，而需要进一步考虑从当前时刻开始能够获得的累计奖励，这个累计奖励通常被称为回报Return。
###### 假设智能体在时间步$t$之后依次获得奖励$r_t,r_{t+1},r_{t+2}$，那么对于一个在时间步$T$结束的任务，从时间步$t$开始的回报可以表示为：
$$G_t = \sum_{k=t}^{T}r_k$$
###### 在长期决策任务中，如果不对未来奖励进行折扣，那么距离当前较远的奖励也会对当前动作产生同等影响，这可能增加训练中的信用分配难度。因此，强化学习通常引入折扣因子$\gamma$，用于控制未来奖励对当前决策的影响程度。需要注意的是，对于语言模型这种通常在完整回答结束后才获得奖励的任务，是否进行折扣以及折扣程度，需要根据具体的奖励设计来决定：
$$G_t = \sum_{k=t}^{T}\gamma^{k-t}r_k$$
###### 其中$\gamma$的取值通常位于0和1之间。当$\gamma$较小时，模型更加关注眼前的奖励；当$\gamma$较大时，模型会更加重视长期收益。在语言模型中，一段回答最后得到的奖励可能会受到前面很多个Token的共同影响，因此这种长期回报的概念尤其重要。
###### 仅仅知道一条轨迹最终获得了多少回报还不够，因为智能体还需要判断：在某个状态下，采取某个动作究竟有多大价值获得多少回报。于是便引出了价值函数。状态价值函数$V^\pi(s)$表示在遵循策略$\pi$的情况下，从状态$s$开始行动所能获得的期望回报；动作价值函数$Q^\pi(s,a)$则进一步考虑了在状态$s$下先采取动作$a$之后能够获得的期望回报。
$$V^\pi(s)=\mathbb{E}_\pi[G_t\mid s_t=s]$$
$$Q^\pi(s,a)=\mathbb{E}_\pi[G_t\mid s_t=s,a_t=a]$$
###### 直观来说，$V$回答的是“处在这个状态本身有多好”，而$Q$回答的是“在这个状态下采取这个动作有多好”。在后续的策略优化中，模型可以根据这些价值信息判断哪些动作值得保留，哪些动作应该被降低概率。

##### ***优势函数（Advantage Function）***
###### 价值函数可以告诉我们当前状态总体上有多好，但它不能直接说明当前采取的某个动作究竟是好还是坏。于是我们引入优势函数，用来衡量某个动作相对于当前状态平均水平的表现：
$$A^\pi(s,a)=Q^\pi(s,a)-V^\pi(s)$$
###### 其中，$Q^\pi(s,a)$表示在状态$s$下采取动作$a$后能够获得的期望回报，$V^\pi(s)$表示在状态$s$下按照当前策略行动能够获得的平均期望回报。因此，优势函数本质上是在比较“采取这个动作”和“通常情况下”的差异。
###### 在实际训练中，$Q^\pi(s,a)$通常可以用一次采样得到的回报$G_t$近似，所以优势函数也常写作：
$$\hat{A}_t=G_t-V(s_t)$$
###### 当$\hat{A}_t>0$时，说明当前动作比预期更好，应该提高它的概率；当$\hat{A}_t<0$时，说明当前动作比预期更差，应该降低它的概率。相比直接使用回报$G_t$，优势函数关注的是动作相对于基准水平的好坏，因此能够减少训练过程中的波动。

##### ***策略与策略梯度***
###### 在强化学习中，智能体最终需要学习的是一个策略Policy，也就是在不同状态下应该如何选择动作。对于语言模型来说，策略实际上就是模型本身：给定当前的上下文之后，模型会通过输出层得到词表上的概率分布，再按照这个分布选择下一个Token。
###### 如果一个动作最终带来了较高的回报，那么我们希望模型以后在类似状态下更倾向于选择这个动作；如果一个动作带来了较低的回报，那么我们则希望降低它再次被选择的概率。这样一来，强化学习的目标就可以理解为：不断调整策略，使高回报动作的概率增大，使低回报动作的概率减小。所以强化学习的优化目标就是最大化期望回报：
$$J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}[R(\tau)]$$
###### 其中，$\theta$表示策略模型的参数，$\tau$表示一整条行动轨迹，$R(\tau)$表示这条轨迹最终能获得的总奖励，$J(\theta)$表示当前策略平均能获得多少回报，所以目标是找到一组参数，让模型平均获得的奖励最大。
###### 但由于动作通常是从概率分布中采样得到的，采样结果本身是离散的，不能像普通神经网络输出那样直接对动作编号进行反向传播。因此，策略梯度并不直接修改已经采样出来的动作，而是利用动作的对数概率和它得到的回报来构造训练信号。
$$\nabla_\theta J(\theta) =\mathbb{E} [R(a) \nabla_\theta \log \pi_\theta(a\mid s)]$$
###### 其中，$\pi_\theta(a\mid s)$表示当前策略在状态$s$下选择动作$a$的概率，$R$表示这次行动得到的回报。当$R$较大时，梯度会推动模型提高该动作的概率；当$R$较小时，模型则会降低该动作的概率。
###### 对于语言模型来说，一整段回答可以看作一条由多个Token组成的轨迹。假设模型生成了$y_1,y_2,\cdots,y_T$，那么整段回答的概率可以分解为：
$$\log \pi_\theta(y\mid x)=\sum_{t=1}^{T}\log \pi_\theta(y_t\mid x,y_{<t})$$
###### 因此，策略梯度可以利用整段回答的奖励，对生成过程中的多个Token概率产生影响。这里的“反向调整”并不是修改已经生成的Token，而是更新模型参数，使模型下一次面对类似上下文时，更倾向于生成能够带来高回报的Token。

In [2]:
# 为进一步学习RL，接下来将使用Python实现上述一些操作
# 两臂老虎机
import torch
import torch.nn as nn
from torch.distributions import Categorical

class BanditPolicy(nn.Module):
    def __init__(self, num_actions):
        super().__init__()
        # 两臂老虎机没有复杂的策略，所以可以将策略简化为两个可训练的数字
        self.num_actions = num_actions
        self.logits = nn.Parameter(torch.zeros(num_actions))

    def forward(self):
        # 返回两个动作的概率
        return torch.softmax(self.logits, dim=0)
    
policy = BanditPolicy(2)
optimizer = torch.optim.SGD(policy.parameters(), lr=0.1)
reward_table = torch.tensor([1.0, 0.0])

# 为了得到负反馈，添加优势函数
baseline = 0.0
baseline_decay = 0.9

for step in range(1000):
    # 每轮开始前清空上一轮的梯度
    optimizer.zero_grad()

    # 当前策略给出的动作概率
    probs = policy()
    dist = Categorical(probs=probs)

    # 根据当前策略采样动作
    action = dist.sample()

    # 得到这个动作的对数概率
    log_prob = dist.log_prob(action)

    # 环境根据动作返回奖励
    reward = reward_table[action]

    # 策略梯度损失
    advantage = reward - baseline
    loss = -advantage * log_prob

    # 反向传播并更新参数
    loss.backward()
    optimizer.step()
    baseline = (
        baseline_decay * baseline + (1-baseline_decay) * reward.item()
    )
    if step % 100 == 0:
        print(
            "step:", step,
            "probs:", policy().detach(),
            "action:", action.item(),
            "reward:", reward.item(),
            "advantage:", advantage.item()
        )


step: 0 probs: tensor([0.5250, 0.4750]) action: 0 reward: 1.0 advantage: 1.0
step: 100 probs: tensor([0.9143, 0.0857]) action: 0 reward: 1.0 advantage: 0.008150279521942139
step: 200 probs: tensor([0.9581, 0.0419]) action: 0 reward: 1.0 advantage: 0.004901111125946045
step: 300 probs: tensor([0.9804, 0.0196]) action: 0 reward: 1.0 advantage: 0.005825400352478027
step: 400 probs: tensor([0.9908, 0.0092]) action: 0 reward: 1.0 advantage: 0.012445509433746338
step: 500 probs: tensor([0.9924, 0.0076]) action: 0 reward: 1.0 advantage: 0.007976949214935303
step: 600 probs: tensor([0.9938, 0.0062]) action: 0 reward: 1.0 advantage: 4.708766937255859e-06
step: 700 probs: tensor([0.9949, 0.0051]) action: 0 reward: 1.0 advantage: 3.331899642944336e-05
step: 800 probs: tensor([0.9958, 0.0042]) action: 0 reward: 1.0 advantage: 0.07289999723434448
step: 900 probs: tensor([0.9958, 0.0042]) action: 0 reward: 1.0 advantage: 1.9073486328125e-06


In [4]:
# 上一个例子仅仅使用了单步策略，下面写多步策略
class TwoStepPolicy(nn.Module):
    def __init__(self, num_steps, num_actions):
        super().__init__()
        self.num_steps = num_steps
        self.num_actions = num_actions

        # 每个时间步都需要一组单独的logits，所以logits的形状应该是[num_steps, num_actions]
        self.logits = nn.Parameter(torch.zeros([num_steps, num_actions]))

    def forward(self, step):
        # 返回指定时间步的动作概率
        return torch.softmax(self.logits[step], dim=0)
policy = TwoStepPolicy(num_steps=2, num_actions=2)
optimizer = torch.optim.SGD(policy.parameters(), lr=0.1)
target_actions = [0, 0]

# 为了得到负反馈，加上优势函数
baselines = [0.0, 0.0]
baseline_decay = 0.9

for episode in range(1000):
    optimizer.zero_grad()
    # 每个episode都必须重新创建动作列表
    actions = []
    log_probs = []

    for step in range(2):
        probs = policy(step)
        dist = Categorical(probs=probs)

        # 根据当前时间步的策略采样动作
        action = dist.sample()

        # 记录这个动作在当前策略下的对数概率
        log_prob = dist.log_prob(action)

        actions.append(action)
        log_probs.append(log_prob)
    # 循环后给最终奖励
    reward = 1.0 if all(
        action.item() == target_actions[step] for step, action in enumerate(actions)
    )else 0.0

    # 即第0步没有即时奖励，第1步获得最终奖励，也就是规定最后一个时间步才获得奖励
    rewards = [0.0, reward]
    gamma = 0.9
    returns = []
    G = 0.0

    # 从后往前计算回报
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G) # 头插法

    advantages = [
        G_t - baseline_t for G_t, baseline_t in zip(returns, baselines)
    ]
    # 使用每个时间步对应的回报计算损失
    loss = sum(
        -A_t * log_prob_t for A_t, log_prob_t in zip(advantages, log_probs)
    )
    loss.backward()
    optimizer.step()

    # 参数更新完后再更新两个时间步的基线
    for step in range(2):
        baselines[step] = (
            baseline_decay * baselines[step] + (1 - baseline_decay) * returns[step]
        )
    if episode % 100 == 0:
        print("actions:", actions)
        print("returns:", returns)
        print("reward:", reward)
        print("loss:", loss)

actions: [tensor(0), tensor(1)]
returns: [0.0, 0.0]
reward: 0.0
loss: tensor(0., grad_fn=<AddBackward0>)
actions: [tensor(0), tensor(0)]
returns: [0.9, 1.0]
reward: 1.0
loss: tensor(0.0358, grad_fn=<AddBackward0>)
actions: [tensor(0), tensor(0)]
returns: [0.9, 1.0]
reward: 1.0
loss: tensor(0.0089, grad_fn=<AddBackward0>)
actions: [tensor(0), tensor(0)]
returns: [0.9, 1.0]
reward: 1.0
loss: tensor(0.0017, grad_fn=<AddBackward0>)
actions: [tensor(0), tensor(0)]
returns: [0.9, 1.0]
reward: 1.0
loss: tensor(0.0038, grad_fn=<AddBackward0>)
actions: [tensor(0), tensor(0)]
returns: [0.9, 1.0]
reward: 1.0
loss: tensor(9.9966e-08, grad_fn=<AddBackward0>)
actions: [tensor(0), tensor(0)]
returns: [0.9, 1.0]
reward: 1.0
loss: tensor(0.0001, grad_fn=<AddBackward0>)
actions: [tensor(0), tensor(0)]
returns: [0.9, 1.0]
reward: 1.0
loss: tensor(0.0003, grad_fn=<AddBackward0>)
actions: [tensor(0), tensor(0)]
returns: [0.9, 1.0]
reward: 1.0
loss: tensor(1.8519e-05, grad_fn=<AddBackward0>)
actions: [tenso